# Pre-Planting Model Preparation

This notebook prepares candidate predictors for a model that estimates crop yield before the planting year begins. Only information available before that year will be considered. The annual climate join by harvest year in the climate exploration notebook is descriptive and will not be used as a predictor here.

## 1. Load the climate summaries and yield records

Load the saved state-year climate summaries and prepared HarvestStat dataset. Keep the eight core crops with non-missing yield values for this initial preparation.

In [2]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent

annual_climate_path = (
    project_root / "data" / "processed" / "climate"
    / "nigeria_state_annual_climate.csv"
)
harveststat_path = project_root / "data" / "processed" / "harveststat_nigeria.csv"

annual_state = pd.read_csv(annual_climate_path)
harveststat = pd.read_csv(harveststat_path)

core_crops = [
    "Maize", "Rice", "Cowpea", "Cassava",
    "Groundnuts (In Shell)", "Yams", "Sorghum", "Millet",
]

crop_targets = harveststat.loc[
    harveststat["product"].isin(core_crops) & harveststat["yield"].notna()
].copy()

print("Annual climate rows:", len(annual_state))
print("Climate years:", annual_state["year"].min(), "to", annual_state["year"].max())
print("Climate states:", annual_state["fnid"].nunique())
print("Core-crop valid-yield records:", len(crop_targets))
print("Target crops:", sorted(crop_targets["product"].unique()))
print(
    "Planting years:",
    crop_targets["planting_year"].min(),
    "to",
    crop_targets["planting_year"].max()
)

Annual climate rows: 925
Climate years: 1999 to 2023
Climate states: 37
Core-crop valid-yield records: 6004
Target crops: ['Cassava', 'Cowpea', 'Groundnuts (In Shell)', 'Maize', 'Millet', 'Rice', 'Sorghum', 'Yams']
Planting years: 1999 to 2023


## 2. Check availability of pre-planting climate history

For each planting year, create candidate features from the previous calendar year and from the preceding three years. Do not include climate from the target planting year. Count how many crop-yield records have complete climate history for each option.

In [3]:
annual_features = annual_state.sort_values(["fnid", "year"]).copy()

if annual_features.duplicated(["fnid", "year"]).any():
    raise ValueError("Annual climate data has duplicate state-year rows.")

for variable, short_name in [
    ("annual_rainfall_mm", "rainfall"),
    ("T2M_annual_mean", "T2M"),
]:
    annual_features[f"prior1_{short_name}"] = (
        annual_features.groupby("fnid")[variable].shift(1)
    )
    annual_features[f"prior3y_mean_{short_name}"] = (
        annual_features.groupby("fnid")[variable]
        .transform(lambda values: values.shift(1).rolling(3, min_periods=3).mean())
    )

features_for_planting = annual_features.rename(columns={"year": "planting_year"})

candidate_rows = crop_targets.merge(
    features_for_planting[
        [
            "fnid",
            "admin_1",
            "planting_year",
            "prior1_rainfall",
            "prior1_T2M",
            "prior3y_mean_rainfall",
            "prior3y_mean_T2M",
        ]
    ],
    on=["fnid", "admin_1", "planting_year"],
    how="left",
    validate="many_to_one",
)

candidate_rows["has_prior1_features"] = candidate_rows[
    ["prior1_rainfall", "prior1_T2M"]
].notna().all(axis=1)

candidate_rows["has_prior3y_features"] = candidate_rows[
    ["prior3y_mean_rainfall", "prior3y_mean_T2M"]
].notna().all(axis=1)

coverage_by_crop = (
    candidate_rows.groupby("product", as_index=False)
    .agg(
        yield_records=("yield", "size"),
        records_with_prior1_features=("has_prior1_features", "sum"),
        records_with_prior3y_features=("has_prior3y_features", "sum"),
    )
)

print(coverage_by_crop.to_string(index=False))
print("\nOverall:")
print("Yield records:", len(candidate_rows))
print("With previous-year climate:", int(candidate_rows["has_prior1_features"].sum()))
print("With previous-three-year climate:", int(candidate_rows["has_prior3y_features"].sum()))

              product  yield_records  records_with_prior1_features  records_with_prior3y_features
              Cassava            844                           813                            749
               Cowpea            838                           806                            740
Groundnuts (In Shell)            768                           735                            667
                Maize            919                           882                            808
               Millet            499                           479                            439
                 Rice            906                           871                            798
              Sorghum            536                           515                            473
                 Yams            694                           666                            610

Overall:
Yield records: 6004
With previous-year climate: 5767
With previous-three-year climate: 5284


## 3. Check records without previous-year climate

The climate archive begins in 1999, so targets from the first year may not have a preceding year of climate data. Identify all records without the prior-year feature before creating the initial candidate table.

In [4]:
missing_prior1 = candidate_rows.loc[~candidate_rows["has_prior1_features"]]

print("Records lacking previous-year climate:", len(missing_prior1))
print("By planting year:")
print(
    missing_prior1.groupby("planting_year")
    .size()
    .rename("records")
    .to_string()
)

Records lacking previous-year climate: 237
By planting year:
planting_year
1999    237


## 4. Create the initial pre-planting candidate table

Keep records with prior-year rainfall and temperature. Retain crop and state identifiers, target yield, and QC flag for review. Do not include area or production as predictors. This is a candidate table for later modeling review, not a trained-model dataset.

In [5]:
preplant_candidate = candidate_rows.loc[
    candidate_rows["has_prior1_features"]
].copy()

preplant_candidate["climate_data_through_year"] = (
    preplant_candidate["planting_year"] - 1
)

preplant_candidate = preplant_candidate[
    [
        "fnid",
        "admin_1",
        "product",
        "season_name",
        "planting_year",
        "harvest_year",
        "yield",
        "qc_flag",
        "climate_data_through_year",
        "prior1_rainfall",
        "prior1_T2M",
    ]
].rename(columns={
    "yield": "target_yield",
    "prior1_rainfall": "prior_year_rainfall_mm",
    "prior1_T2M": "prior_year_T2M_C",
})

print("Candidate rows:", len(preplant_candidate))
print("States:", preplant_candidate["fnid"].nunique())
print("Crops:", preplant_candidate["product"].nunique())
print("Missing feature values:")
print(
    preplant_candidate[
        ["prior_year_rainfall_mm", "prior_year_T2M_C"]
    ].isna().sum().to_string()
)
print("QC flag counts:")
print(preplant_candidate["qc_flag"].value_counts().sort_index().to_string())
print("Columns:", preplant_candidate.columns.tolist())
print(preplant_candidate.head(10).to_string(index=False))

output_path = (
    project_root
    / "data"
    / "processed"
    / "climate"
    / "nigeria_preplant_candidate.csv"
)
preplant_candidate.to_csv(output_path, index=False)
print("Saved:", output_path)

Candidate rows: 5767
States: 37
Crops: 8
Missing feature values:
prior_year_rainfall_mm    0
prior_year_T2M_C          0
QC flag counts:
qc_flag
0    5329
1      16
2     422
Columns: ['fnid', 'admin_1', 'product', 'season_name', 'planting_year', 'harvest_year', 'target_yield', 'qc_flag', 'climate_data_through_year', 'prior_year_rainfall_mm', 'prior_year_T2M_C']
      fnid admin_1 product season_name  planting_year  harvest_year  target_yield  qc_flag  climate_data_through_year  prior_year_rainfall_mm  prior_year_T2M_C
NG1996A101    Abia Cassava      Annual           2000          2000     17.494752        0                       1999             2584.352455         25.400466
NG1996A101    Abia Cassava      Annual           2001          2001     16.470588        0                       2000             2231.330500         24.984044
NG1996A101    Abia Cassava      Annual           2002          2002     16.470588        0                       2001             2113.211151         24.99

## 5. Review a possible chronological evaluation split

Count records by crop and planting year, then review a proposed split: training on 2000–2018, validation on 2019–2020, and a candidate later period of 2021–2023. These were candidate periods at the time of preparation. Notebook 04 subsequently examined 2021–2023, so it is no longer an untouched test period. Do not randomly split the rows.

In [6]:
candidate_rows["proposed_split"] = "outside proposed periods"

candidate_rows.loc[
    candidate_rows["planting_year"].between(2000, 2018),
    "proposed_split"
] = "train 2000-2018"

candidate_rows.loc[
    candidate_rows["planting_year"].between(2019, 2020),
    "proposed_split"
] = "validation 2019-2020"

candidate_rows.loc[
    candidate_rows["planting_year"].between(2021, 2023),
    "proposed_split"
] = "test 2021-2023"

print("Records by proposed period:")
print(candidate_rows["proposed_split"].value_counts().to_string())

print("\nRecords by crop and proposed period:")
print(
    candidate_rows.groupby(["product", "proposed_split"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

Records by proposed period:
proposed_split
train 2000-2018             4558
test 2021-2023               726
validation 2019-2020         483
outside proposed periods     237

Records by crop and proposed period:
proposed_split         outside proposed periods  test 2021-2023  train 2000-2018  validation 2019-2020
product                                                                                               
Cassava                                      31             102              643                    68
Cowpea                                       32             111              621                    74
Groundnuts (In Shell)                        33              87              591                    57
Maize                                        37             111              697                    74
Millet                                       20              60              379                    40
Rice                                         35             111   

## 6. Review QC flags across the proposed time splits

Count target records by QC flag within each proposed split and crop. Use this to decide how to handle flagged yields in the main analysis and sensitivity checks; do not remove them automatically.

In [7]:
eligible_rows = candidate_rows.loc[
    candidate_rows["has_prior1_features"]
].copy()

print("QC flags by proposed split:")
print(
    eligible_rows.groupby(["proposed_split", "qc_flag"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

print("\nQC flags by crop and proposed split:")
qc_by_crop_split = (
    eligible_rows.groupby(["product", "proposed_split"])
    .agg(
        records=("yield", "size"),
        flagged_records=("qc_flag", lambda values: int((values != 0).sum())),
    )
    .reset_index()
)
print(qc_by_crop_split.to_string(index=False))

QC flags by proposed split:
qc_flag                  0   1    2
proposed_split                     
test 2021-2023         679   0   47
train 2000-2018       4269  16  273
validation 2019-2020   381   0  102

QC flags by crop and proposed split:
              product       proposed_split  records  flagged_records
              Cassava       test 2021-2023      102               12
              Cassava      train 2000-2018      643               62
              Cassava validation 2019-2020       68               30
               Cowpea       test 2021-2023      111                5
               Cowpea      train 2000-2018      621               22
               Cowpea validation 2019-2020       74               10
Groundnuts (In Shell)       test 2021-2023       87                5
Groundnuts (In Shell)      train 2000-2018      591               31
Groundnuts (In Shell) validation 2019-2020       57               12
                Maize       test 2021-2023      111             

## 7. Locate QC-2 records in the validation period

Summarize QC flags by planting year and identify which crops contribute QC-2 records in the validation years. Use this to understand the validation-set composition; do not change the chronological split or remove records based on this check alone.

In [8]:
qc_by_year = (
    eligible_rows.groupby(["planting_year", "qc_flag"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1, 2], fill_value=0)
)

qc_by_year["total_records"] = qc_by_year[[0, 1, 2]].sum(axis=1)
qc_by_year["flagged_pct"] = (
    100 * (qc_by_year[1] + qc_by_year[2]) / qc_by_year["total_records"]
)

print(qc_by_year.round(1).to_string())

validation_qc2 = eligible_rows.loc[
    eligible_rows["planting_year"].between(2019, 2020)
    & (eligible_rows["qc_flag"] == 2)
]

print("\nValidation-period QC-2 records by crop and year:")
print(
    validation_qc2.groupby(["product", "planting_year"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

qc_flag          0  1   2  total_records  flagged_pct
planting_year                                        
2000           232  0   9            241          3.7
2001           221  0  21            242          8.7
2002           211  0  29            240         12.1
2003           208  0  31            239         13.0
2004           203  0  29            232         12.5
2005           213  0  18            231          7.8
2006           214  2  18            234          8.5
2007           217  1  13            231          6.1
2008           222  0   9            231          3.9
2009           232  0   7            239          2.9
2010           228  5   5            238          4.2
2011           234  0   3            237          1.3
2012           229  8   3            240          4.6
2013           241  0   2            243          0.8
2014           246  0   0            246          0.0
2015           248  0   0            248          0.0
2016           250  0   2   

## 8. Finalize the pre-planting candidate table and time splits

Exclude records without prior-year climate. Keep all remaining yield records in the main candidate table, retain QC flags for review, and assign the proposed chronological periods by planting year. Keep identifiers and review fields separate from the climate predictors.

In [9]:
preplant_model_data = candidate_rows.loc[
    candidate_rows["has_prior1_features"]
].copy()

preplant_model_data["climate_data_through_year"] = (
    preplant_model_data["planting_year"] - 1
)

preplant_model_data = preplant_model_data[
    [
        "fnid",
        "admin_1",
        "product",
        "season_name",
        "planting_year",
        "harvest_year",
        "yield",
        "qc_flag",
        "climate_data_through_year",
        "prior1_rainfall",
        "prior1_T2M",
        "proposed_split",
    ]
].rename(columns={
    "yield": "target_yield",
    "prior1_rainfall": "prior_year_rainfall_mm",
    "prior1_T2M": "prior_year_T2M_C",
})

print("Rows:", len(preplant_model_data))
print("Missing prior-year climate values:")
print(
    preplant_model_data[
        ["prior_year_rainfall_mm", "prior_year_T2M_C"]
    ].isna().sum().to_string()
)
print("\nRows by time split:")
print(preplant_model_data["proposed_split"].value_counts().to_string())
print("\nQC flags by time split:")
print(
    preplant_model_data.groupby(["proposed_split", "qc_flag"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)
print("\nArea/production included:", bool({"area", "production"} & set(preplant_model_data.columns)))

output_path = (
    project_root
    / "data"
    / "processed"
    / "climate"
    / "nigeria_preplant_model_table.csv"
)
preplant_model_data.to_csv(output_path, index=False)
print("\nSaved:", output_path)

Rows: 5767
Missing prior-year climate values:
prior_year_rainfall_mm    0
prior_year_T2M_C          0

Rows by time split:
proposed_split
train 2000-2018         4558
test 2021-2023           726
validation 2019-2020     483

QC flags by time split:
qc_flag                  0   1    2
proposed_split                     
test 2021-2023         679   0   47
train 2000-2018       4269  16  273
validation 2019-2020   381   0  102

Area/production included: False

Saved: /Users/davidoffor/Desktop/crop-yield-prediction/data/processed/climate/nigeria_preplant_model_table.csv


## 9. Establish validation baselines

Create two simple yield predictors using only the training period: the historical median for each crop, and the historical median for each crop–state combination. If a crop–state combination is missing from training, use that crop’s training median. Evaluate both on the validation period by MAE, RMSE, and R². The later 2021–2023 period has since been examined in this project and is not available as an untouched test for new model selection.

In [10]:
import numpy as np
import pandas as pd

train = preplant_model_data.loc[
    preplant_model_data["proposed_split"] == "train 2000-2018"
].copy()

validation = preplant_model_data.loc[
    preplant_model_data["proposed_split"] == "validation 2019-2020"
].copy()

# Learn both baseline predictions using training records only.
overall_median = train["target_yield"].median()
crop_medians = train.groupby("product")["target_yield"].median()
crop_state_medians = train.groupby(
    ["product", "fnid"]
)["target_yield"].median()

validation["pred_crop_median"] = (
    validation["product"].map(crop_medians).fillna(overall_median)
)

state_predictions = []
for product, fnid in zip(validation["product"], validation["fnid"]):
    state_predictions.append(
        crop_state_medians.get(
            (product, fnid),
            crop_medians.get(product, overall_median)
        )
    )

validation["pred_crop_state_median"] = state_predictions

unseen_pairs = sum(
    (product, fnid) not in crop_state_medians.index
    for product, fnid in zip(validation["product"], validation["fnid"])
)
print("Validation records using crop-only fallback:", unseen_pairs)

def regression_scores(observed, predicted):
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    mae = np.mean(np.abs(observed - predicted))
    rmse = np.sqrt(np.mean((observed - predicted) ** 2))
    r2 = 1 - np.sum((observed - predicted) ** 2) / np.sum(
        (observed - observed.mean()) ** 2
    )

    return mae, rmse, r2

score_rows = []

for group_name, group in [
    ("Overall", validation),
    *list(validation.groupby("product")),
]:
    for method, prediction_column in [
        ("Crop median", "pred_crop_median"),
        ("Crop-state median", "pred_crop_state_median"),
    ]:
        mae, rmse, r2 = regression_scores(
            group["target_yield"],
            group[prediction_column],
        )
        score_rows.append({
            "group": group_name,
            "baseline": method,
            "records": len(group),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        })

baseline_scores = pd.DataFrame(score_rows)
print(baseline_scores.round(3).to_string(index=False))

Validation records using crop-only fallback: 1
                group          baseline  records   MAE  RMSE     R2
              Overall       Crop median      483 1.179 2.300  0.503
              Overall Crop-state median      483 1.304 2.562  0.384
              Cassava       Crop median       68 4.058 4.804 -1.184
              Cassava Crop-state median       68 4.627 5.434 -1.793
               Cowpea       Crop median       74 0.271 0.380 -0.360
               Cowpea Crop-state median       74 0.294 0.434 -0.768
Groundnuts (In Shell)       Crop median       57 0.306 0.436 -0.165
Groundnuts (In Shell) Crop-state median       57 0.346 0.460 -0.295
                Maize       Crop median       74 0.452 0.557 -0.979
                Maize Crop-state median       74 0.468 0.598 -1.276
               Millet       Crop median       40 0.279 0.468 -0.173
               Millet Crop-state median       40 0.352 0.495 -0.312
                 Rice       Crop median       74 0.269 0.368 -0.001
 

## 10. Compare validation models with and without climate

Fit a crop-and-state model, then fit the same model with prior-year rainfall and temperature added. Train both using 2000–2018 only and evaluate on 2019–2020. Compare them with the crop-median baseline. Use per-crop MAE and RMSE as the main comparison; report pooled metrics as secondary. At the time this experiment was planned, 2021–2023 was reserved; notebook 04 later examined those years, so they are not an untouched test for future model selection.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

train = preplant_model_data.loc[
    preplant_model_data["proposed_split"] == "train 2000-2018"
].copy()

validation = preplant_model_data.loc[
    preplant_model_data["proposed_split"] == "validation 2019-2020"
].copy()

categorical_features = ["product", "fnid"]
climate_features = ["prior_year_rainfall_mm", "prior_year_T2M_C"]

# Crop-only median baseline, fit on training records only.
crop_medians = train.groupby("product")["target_yield"].median()
overall_median = train["target_yield"].median()
predictions = {
    "Crop median": validation["product"].map(crop_medians).fillna(overall_median).to_numpy()
}

def make_random_forest(numeric_features):
    all_features = categorical_features + numeric_features

    preprocessing = ColumnTransformer(
        transformers=[
            ("categories", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("numeric", "passthrough", numeric_features),
        ]
    )

    return Pipeline([
        ("preprocessing", preprocessing),
        ("model", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        )),
    ])

# Compare crop-and-state information alone with crop, state, and prior-year climate.
for model_name, numeric_features in [
    ("Crop and state", []),
    ("Crop, state, and climate", climate_features),
]:
    model = make_random_forest(numeric_features)
    model.fit(
        train[categorical_features + numeric_features],
        train["target_yield"],
    )
    predictions[model_name] = model.predict(
        validation[categorical_features + numeric_features]
    )

score_rows = []

for model_name, predicted in predictions.items():
    for group_name, row_indices in [
        ("Overall", validation.index),
        *[(crop, group.index) for crop, group in validation.groupby("product")],
    ]:
        observed = validation.loc[row_indices, "target_yield"]
        group_predictions = predicted[validation.index.get_indexer(row_indices)]

        score_rows.append({
            "group": group_name,
            "model": model_name,
            "records": len(observed),
            "MAE": mean_absolute_error(observed, group_predictions),
            "RMSE": np.sqrt(mean_squared_error(observed, group_predictions)),
            "R2": r2_score(observed, group_predictions),
        })

validation_model_scores = pd.DataFrame(score_rows)
print(validation_model_scores.round(3).to_string(index=False))

                group                    model  records   MAE  RMSE     R2
              Overall              Crop median      483 1.179 2.300  0.503
              Cassava              Crop median       68 4.058 4.804 -1.184
               Cowpea              Crop median       74 0.271 0.380 -0.360
Groundnuts (In Shell)              Crop median       57 0.306 0.436 -0.165
                Maize              Crop median       74 0.452 0.557 -0.979
               Millet              Crop median       40 0.279 0.468 -0.173
                 Rice              Crop median       74 0.269 0.368 -0.001
              Sorghum              Crop median       44 0.160 0.203 -0.000
                 Yams              Crop median       52 3.546 4.207 -0.160
              Overall           Crop and state      483 1.306 2.605  0.363
              Cassava           Crop and state       68 4.481 5.493 -1.855
               Cowpea           Crop and state       74 0.276 0.377 -0.337
Groundnuts (In Shell)    

### Initial validation result

The crop-median baseline performed best overall. Adding crop/state categories and prior-year climate did not produce a consistent improvement by crop. Treat this as an initial result: the validation set has a higher share of QC-2 records than training and the later evaluation period. A QC-0-only sensitivity comparison checks whether the result changes when restricted to records with QC flag 0. The 2021–2023 period was subsequently examined in this project and is not an untouched test set.

In [12]:
train = preplant_model_data.loc[
    (preplant_model_data["proposed_split"] == "train 2000-2018")
    & (preplant_model_data["qc_flag"] == 0)
].copy()

validation = preplant_model_data.loc[
    (preplant_model_data["proposed_split"] == "validation 2019-2020")
    & (preplant_model_data["qc_flag"] == 0)
].copy()

In [13]:
print(validation_model_scores.round(3).to_string(index=False))

                group                    model  records   MAE  RMSE     R2
              Overall              Crop median      483 1.179 2.300  0.503
              Cassava              Crop median       68 4.058 4.804 -1.184
               Cowpea              Crop median       74 0.271 0.380 -0.360
Groundnuts (In Shell)              Crop median       57 0.306 0.436 -0.165
                Maize              Crop median       74 0.452 0.557 -0.979
               Millet              Crop median       40 0.279 0.468 -0.173
                 Rice              Crop median       74 0.269 0.368 -0.001
              Sorghum              Crop median       44 0.160 0.203 -0.000
                 Yams              Crop median       52 3.546 4.207 -0.160
              Overall           Crop and state      483 1.306 2.605  0.363
              Cassava           Crop and state       68 4.481 5.493 -1.855
               Cowpea           Crop and state       74 0.276 0.377 -0.337
Groundnuts (In Shell)    

This is the all-record table: it still has 483 validation records. The QC-0-only validation set should have 381. It looks like the updated filters weren’t used to refit and score the models.

We need to create a separate QC-0-only result table:

In [14]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

train_qc0 = preplant_model_data.loc[
    (preplant_model_data["proposed_split"] == "train 2000-2018")
    & (preplant_model_data["qc_flag"] == 0)
].copy().reset_index(drop=True)

validation_qc0 = preplant_model_data.loc[
    (preplant_model_data["proposed_split"] == "validation 2019-2020")
    & (preplant_model_data["qc_flag"] == 0)
].copy().reset_index(drop=True)

print("QC-0 training records:", len(train_qc0))
print("QC-0 validation records:", len(validation_qc0))

categorical_features = ["product", "fnid"]
climate_features = ["prior_year_rainfall_mm", "prior_year_T2M_C"]

crop_medians_qc0 = train_qc0.groupby("product")["target_yield"].median()
overall_median_qc0 = train_qc0["target_yield"].median()

predictions_qc0 = {
    "Crop median": (
        validation_qc0["product"]
        .map(crop_medians_qc0)
        .fillna(overall_median_qc0)
        .to_numpy()
    )
}

def make_qc0_random_forest(numeric_features):
    preprocessing = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ])

    return Pipeline([
        ("preprocessing", preprocessing),
        ("model", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        )),
    ])

for model_name, numeric_features in [
    ("Crop and state", []),
    ("Crop, state, and climate", climate_features),
]:
    model = make_qc0_random_forest(numeric_features)
    model.fit(
        train_qc0[categorical_features + numeric_features],
        train_qc0["target_yield"],
    )
    predictions_qc0[model_name] = model.predict(
        validation_qc0[categorical_features + numeric_features]
    )

score_rows_qc0 = []

groups = [("Overall", validation_qc0)]
groups += list(validation_qc0.groupby("product"))

for model_name, predicted in predictions_qc0.items():
    for group_name, group in groups:
        positions = group.index.to_numpy()
        observed = group["target_yield"]
        group_predictions = predicted[positions]

        score_rows_qc0.append({
            "group": group_name,
            "model": model_name,
            "records": len(group),
            "MAE": mean_absolute_error(observed, group_predictions),
            "RMSE": np.sqrt(mean_squared_error(observed, group_predictions)),
            "R2": r2_score(observed, group_predictions),
        })

qc0_validation_scores = pd.DataFrame(score_rows_qc0)
print(qc0_validation_scores.round(3).to_string(index=False))

QC-0 training records: 4269
QC-0 validation records: 381
                group                    model  records   MAE  RMSE     R2
              Overall              Crop median      381 0.905 1.901  0.669
              Cassava              Crop median       38 3.340 4.322 -0.973
               Cowpea              Crop median       64 0.290 0.406 -0.448
Groundnuts (In Shell)              Crop median       45 0.318 0.457 -0.346
                Maize              Crop median       54 0.427 0.535 -1.068
               Millet              Crop median       36 0.292 0.491 -0.250
                 Rice              Crop median       74 0.269 0.368 -0.001
              Sorghum              Crop median       32 0.143 0.173 -0.005
                 Yams              Crop median       38 3.340 4.011 -0.021
              Overall           Crop and state      381 0.979 1.996  0.635
              Cassava           Crop and state       38 3.909 4.645 -1.279
               Cowpea           Crop and st

The QC-0-only check tells a similar story: the crop-median baseline scores better overall (MAE 0.905) than the crop, state, and climate model (MAE 0.949). So the one-year climate features still haven’t shown a broad benefit. The climate model helps slightly for some crops relative to the crop-and-state model, but not consistently relative to the simpler baseline.

Next, compare three-year pre-planting climate averages prepared earlier, using the same development and validation split. The later 2021–2023 period is treated as previously examined in the current project.

## 11. Compare three-year pre-planting climate history

Repeat the validation comparison using average climate over the three complete years before planting. Use the same crop-median and crop-and-state comparisons as before. This checks whether a longer climate history is more useful than previous-year values. The 2021–2023 period has since been examined and must not be treated as an untouched test for additional model selection.

In [15]:
features_3y = candidate_rows.loc[
    candidate_rows["has_prior3y_features"]
].copy()

train_3y = features_3y.loc[
    features_3y["proposed_split"] == "train 2000-2018"
].copy().reset_index(drop=True)

validation_3y = features_3y.loc[
    features_3y["proposed_split"] == "validation 2019-2020"
].copy().reset_index(drop=True)

print("Three-year training records:", len(train_3y))
print("Three-year validation records:", len(validation_3y))

categorical_features = ["product", "fnid"]
climate_features_3y = ["prior3y_mean_rainfall", "prior3y_mean_T2M"]

crop_medians_3y = train_3y.groupby("product")["yield"].median()
overall_median_3y = train_3y["yield"].median()

predictions_3y = {
    "Crop median": (
        validation_3y["product"]
        .map(crop_medians_3y)
        .fillna(overall_median_3y)
        .to_numpy()
    )
}

for model_name, numeric_features in [
    ("Crop and state", []),
    ("Crop, state, and climate", climate_features_3y),
]:
    model = make_random_forest(numeric_features)
    model.fit(
        train_3y[categorical_features + numeric_features],
        train_3y["yield"],
    )
    predictions_3y[model_name] = model.predict(
        validation_3y[categorical_features + numeric_features]
    )

score_rows_3y = []

groups_3y = [("Overall", validation_3y)]
groups_3y += list(validation_3y.groupby("product"))

for model_name, predicted in predictions_3y.items():
    for group_name, group in groups_3y:
        positions = group.index.to_numpy()
        observed = group["yield"]
        group_predictions = predicted[positions]

        score_rows_3y.append({
            "group": group_name,
            "model": model_name,
            "records": len(group),
            "MAE": mean_absolute_error(observed, group_predictions),
            "RMSE": np.sqrt(mean_squared_error(observed, group_predictions)),
            "R2": r2_score(observed, group_predictions),
        })

three_year_validation_scores = pd.DataFrame(score_rows_3y)
print(three_year_validation_scores.round(3).to_string(index=False))

Three-year training records: 4075
Three-year validation records: 483
                group                    model  records   MAE  RMSE     R2
              Overall              Crop median      483 1.164 2.278  0.513
              Cassava              Crop median       68 4.014 4.760 -1.143
               Cowpea              Crop median       74 0.262 0.371 -0.294
Groundnuts (In Shell)              Crop median       57 0.300 0.429 -0.130
                Maize              Crop median       74 0.441 0.546 -0.902
               Millet              Crop median       40 0.280 0.472 -0.193
                 Rice              Crop median       74 0.271 0.369 -0.005
              Sorghum              Crop median       44 0.162 0.203 -0.002
                 Yams              Crop median       52 3.492 4.165 -0.136
              Overall           Crop and state      483 1.302 2.645  0.343
              Cassava           Crop and state       68 4.495 5.588 -1.954
               Cowpea          

The three-year climate model performs better than the one-year climate model on validation, and better than the crop-and-state model. But it still doesn’t beat the crop-median baseline overall: MAE is 1.205 versus 1.164. This suggests the longer climate history may help somewhat, but it hasn’t shown a general benefit yet.

Because QC-2 yields are overrepresented in validation, we'll check the three-year model on QC-0 records too.

### Three-year climate model: initial validation result

The three-year climate model performed better overall than the one-year climate model and the crop-and-state model. The crop-median baseline still had lower overall MAE. Since the validation period contains a higher share of QC-2 yields, repeat the three-year comparison using QC-0 records as a sensitivity analysis. The 2021–2023 period is examined later in this project and must not be treated as an untouched test.

In [16]:
train_3y_qc0 = features_3y.loc[
    (features_3y["proposed_split"] == "train 2000-2018")
    & (features_3y["qc_flag"] == 0)
].copy().reset_index(drop=True)

validation_3y_qc0 = features_3y.loc[
    (features_3y["proposed_split"] == "validation 2019-2020")
    & (features_3y["qc_flag"] == 0)
].copy().reset_index(drop=True)

print("Three-year QC-0 training records:", len(train_3y_qc0))
print("Three-year QC-0 validation records:", len(validation_3y_qc0))

crop_medians_3y_qc0 = train_3y_qc0.groupby("product")["yield"].median()
overall_median_3y_qc0 = train_3y_qc0["yield"].median()

predictions_3y_qc0 = {
    "Crop median": (
        validation_3y_qc0["product"]
        .map(crop_medians_3y_qc0)
        .fillna(overall_median_3y_qc0)
        .to_numpy()
    )
}

for model_name, numeric_features in [
    ("Crop and state", []),
    ("Crop, state, and climate", climate_features_3y),
]:
    model = make_random_forest(numeric_features)
    model.fit(
        train_3y_qc0[categorical_features + numeric_features],
        train_3y_qc0["yield"],
    )
    predictions_3y_qc0[model_name] = model.predict(
        validation_3y_qc0[categorical_features + numeric_features]
    )

score_rows_3y_qc0 = []
groups_3y_qc0 = [("Overall", validation_3y_qc0)]
groups_3y_qc0 += list(validation_3y_qc0.groupby("product"))

for model_name, predicted in predictions_3y_qc0.items():
    for group_name, group in groups_3y_qc0:
        positions = group.index.to_numpy()
        observed = group["yield"]
        group_predictions = predicted[positions]

        score_rows_3y_qc0.append({
            "group": group_name,
            "model": model_name,
            "records": len(group),
            "MAE": mean_absolute_error(observed, group_predictions),
            "RMSE": np.sqrt(mean_squared_error(observed, group_predictions)),
            "R2": r2_score(observed, group_predictions),
        })

qc0_three_year_scores = pd.DataFrame(score_rows_3y_qc0)
print(qc0_three_year_scores.round(3).to_string(index=False))

Three-year QC-0 training records: 3816
Three-year QC-0 validation records: 381
                group                    model  records   MAE  RMSE     R2
              Overall              Crop median      381 0.892 1.877  0.677
              Cassava              Crop median       38 3.257 4.230 -0.890
               Cowpea              Crop median       64 0.283 0.398 -0.393
Groundnuts (In Shell)              Crop median       45 0.308 0.448 -0.291
                Maize              Crop median       54 0.423 0.531 -1.034
               Millet              Crop median       36 0.295 0.497 -0.281
                 Rice              Crop median       74 0.270 0.369 -0.003
              Sorghum              Crop median       32 0.143 0.173 -0.003
                 Yams              Crop median       38 3.316 3.999 -0.015
              Overall           Crop and state      381 0.949 1.947  0.653
              Cassava           Crop and state       38 3.740 4.357 -1.005
               Cowpea

The QC-0 results show a small, mixed signal: the three-year climate model has slightly lower overall MAE than the crop-median baseline (0.873 vs. 0.892), but higher RMSE (1.974 vs. 1.877). In the all-record validation, crop median still had lower MAE than the climate model (1.164 vs. 1.205). So climate hasn’t shown a robust overall advantage.

The following comparison reproduces the previously examined 2021–2023 results: crop-median baseline versus the three-year climate model, trained through 2020. Include a separate QC-0-only sensitivity result. These years must not be treated as an untouched test or used for new model selection.

## 12. Reproduction on the previously examined 2021–2023 period

Fit the selected crop-median baseline and three-year climate model using planting years through 2020, then evaluate on 2021–2023. Also report a pre-planned QC-0-only sensitivity analysis. Do not use test results to change or tune the models.

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

final_train = features_3y.loc[
    features_3y["planting_year"].between(2000, 2020)
].copy()

final_test = features_3y.loc[
    features_3y["proposed_split"] == "test 2021-2023"
].copy()

categorical_features = ["product", "fnid"]
climate_features_3y = ["prior3y_mean_rainfall", "prior3y_mean_T2M"]

def final_comparison(train_frame, test_frame, subset_label):
    train_frame = train_frame.reset_index(drop=True)
    test_frame = test_frame.reset_index(drop=True)

    crop_medians = train_frame.groupby("product")["yield"].median()
    overall_median = train_frame["yield"].median()

    predictions = {
        "Crop median": (
            test_frame["product"]
            .map(crop_medians)
            .fillna(overall_median)
            .to_numpy()
        )
    }

    preprocessing = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", climate_features_3y),
    ])

    climate_model = Pipeline([
        ("preprocessing", preprocessing),
        ("model", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        )),
    ])

    climate_model.fit(
        train_frame[categorical_features + climate_features_3y],
        train_frame["yield"],
    )
    predictions["Crop, state, and climate"] = climate_model.predict(
        test_frame[categorical_features + climate_features_3y]
    )

    rows = []
    groups = [("Overall", test_frame)]
    groups += list(test_frame.groupby("product"))

    for model_name, predicted in predictions.items():
        for group_name, group in groups:
            positions = group.index.to_numpy()
            observed = group["yield"]
            group_predicted = predicted[positions]

            rows.append({
                "subset": subset_label,
                "group": group_name,
                "model": model_name,
                "records": len(group),
                "MAE": mean_absolute_error(observed, group_predicted),
                "RMSE": np.sqrt(mean_squared_error(observed, group_predicted)),
                "R2": r2_score(observed, group_predicted),
            })

    return pd.DataFrame(rows)

print("All-record training rows:", len(final_train))
print("All-record test rows:", len(final_test))

all_test_scores = final_comparison(final_train, final_test, "All eligible yields")

final_train_qc0 = final_train.loc[final_train["qc_flag"] == 0].copy()
final_test_qc0 = final_test.loc[final_test["qc_flag"] == 0].copy()

print("QC-0 training rows:", len(final_train_qc0))
print("QC-0 test rows:", len(final_test_qc0))

qc0_test_scores = final_comparison(
    final_train_qc0,
    final_test_qc0,
    "QC-0 only",
)

final_test_scores = pd.concat(
    [all_test_scores, qc0_test_scores],
    ignore_index=True,
)

print(final_test_scores.round(3).to_string(index=False))

All-record training rows: 4558
All-record test rows: 726
QC-0 training rows: 4197
QC-0 test rows: 679
             subset                 group                    model  records   MAE  RMSE     R2
All eligible yields               Overall              Crop median      726 1.106 2.175  0.609
All eligible yields               Cassava              Crop median      102 3.546 4.307 -0.567
All eligible yields                Cowpea              Crop median      111 0.252 0.358 -0.151
All eligible yields Groundnuts (In Shell)              Crop median       87 0.277 0.390 -0.007
All eligible yields                 Maize              Crop median      111 0.476 0.590 -0.731
All eligible yields                Millet              Crop median       60 0.289 0.430 -0.067
All eligible yields                  Rice              Crop median      111 0.299 0.400 -0.046
All eligible yields               Sorghum              Crop median       66 0.160 0.224 -0.029
All eligible yields                  Yams  

On the all-record test set, the crop-median baseline performs better overall than the climate model:
- MAE: 1.106 vs. 1.146
- RMSE: 2.175 vs. 2.623
- R²: 0.609 vs. 0.431
The climate model helps some crops, such as Groundnuts and Maize, but not consistently across crops. The QC-0-only results were truncated, so let’s retrieve their overall comparison too.

In [18]:
print(
    final_test_scores.loc[
        final_test_scores["group"] == "Overall",
        ["subset", "model", "records", "MAE", "RMSE", "R2"]
    ].round(3).to_string(index=False)
)

             subset                    model  records   MAE  RMSE    R2
All eligible yields              Crop median      726 1.106 2.175 0.609
All eligible yields Crop, state, and climate      726 1.146 2.623 0.431
          QC-0 only              Crop median      679 1.023 2.060 0.651
          QC-0 only Crop, state, and climate      679 1.040 2.361 0.541


The QC-0 sensitivity check confirms the same ranking on the test set: the crop-median baseline beats the climate model on MAE, RMSE, and R². So the three-year climate features did not improve overall pre-planting prediction accuracy in this evaluation. They helped some crops on some metrics, but the gains weren’t consistent.

## Final conclusion: pre-planting climate evaluation

The final chronological test covered planting years 2021–2023. On all eligible yields, the crop-median baseline outperformed the crop/state/three-year-climate model: MAE 1.106 versus 1.146, RMSE 2.175 versus 2.623, and R² 0.609 versus 0.431. The QC-0-only sensitivity check showed the same ranking: MAE 1.023 versus 1.040, RMSE 2.060 versus 2.361, and R² 0.651 versus 0.541.

The climate model improved some crop-level errors, but not consistently. These results do not support claiming that the tested pre-planting climate features improve overall accuracy. The crop-median baseline is the stronger reference model for this evaluation.

The comparison used a chronological split by planting year. Target-year climate, area, and production were not predictors. QC flags were retained as review metadata and were not used as predictors. Any further model changes must not be tuned against these test results.

## Initial model evaluation and conclusion

I evaluated yield predictions using a time-based split. Earlier planting years were used for training, and 2021–2023 were held out for testing. This reflects the intended task: predicting yields for later years using information available before planting.

I compared a simple crop-median baseline with a model using crop, state, and climate information from the previous three years. Production and area were excluded because yield is derived from production divided by area. The QC flag was used to define a sensitivity analysis, not as a model predictor.

On the held-out test data, the climate model did not improve overall performance:

| Records evaluated | Model | MAE | RMSE | R² |
|---|---|---:|---:|---:|
| All eligible records | Crop median baseline | 1.106 | 2.175 | 0.609 |
| All eligible records | Crop, state, and climate | 1.146 | 2.623 | 0.431 |
| QC flag 0 only | Crop median baseline | 1.023 | 2.060 | 0.651 |
| QC flag 0 only | Crop, state, and climate | 1.040 | 2.361 | 0.541 |

Lower MAE and RMSE indicate smaller prediction errors; higher R² indicates that predictions explain more of the variation in yield. By all three overall measures, the crop-median baseline performed better than the climate model in this evaluation. Results varied by crop, so the overall scores should not be taken to mean that every crop behaved the same way.

These are initial results from the available data and chosen features. They do not show that climate is unrelated to yield. They show that the climate features and model tested here did not provide better overall predictions than the simple baseline. The test years have now been examined, so they should not be used repeatedly to choose or tune models and then reported as an untouched final test.